### 4 Exemple 1 : Emploi du temps Universitaire

In [ ]:
import random

# --- 1. DONNÉES ET CONSTANTES ---
NB_COURS = 30
NB_SALLES = 12
NB_CRENEAUX = 15
NB_PROFS = 20

capacites_salles = {i: random.choice([20, 50, 100]) for i in range(NB_SALLES)}
cours_info = {i: (random.randint(15, 90), random.randint(0, NB_PROFS-1)) for i in range(NB_COURS)}

# AJOUT : Préférences des profs (Jour préféré entre 0 et 4)
prefs_profs = {i: random.randint(0, 4) for i in range(NB_PROFS)}

# --- 2. FONCTIONS DE GÉNÉRATION ET CONFLITS ---
def generer_emploi_aleatoire():
    chromosome = []
    for cours_id in range(NB_COURS):
        salle = random.randint(0, NB_SALLES - 1)
        creneau = random.randint(0, NB_CRENEAUX - 1)
        chromosome.append((cours_id, salle, creneau)) 
    return chromosome

def detecter_conflits(chromosome):
    conflits = 0
    occupation_salle = {} 
    occupation_prof = {}
    
    for cours, salle, creneau in chromosome:
        prof_id = cours_info[cours][1]
        
        # Conflit 1 : Salle occupée deux fois
        cle_salle = (salle, creneau)
        if cle_salle in occupation_salle:
            conflits += 1
        occupation_salle[cle_salle] = True
        
        # Conflit 2 : Prof occupé deux fois
        cle_prof = (prof_id, creneau)
        if cle_prof in occupation_prof:
            conflits += 1
        occupation_prof[cle_prof] = True
        
    return conflits

# --- 3. FONCTION FITNESS (Celle que tu as modifiée) ---
def fitness_emploi_du_temps(chromosome):
    score = 1000
    
    # A. Pénalités pour les conflits graves
    nb_conflits = detecter_conflits(chromosome)
    score -= nb_conflits * 50
    
    # B. Salles trop petites
    for cours, salle, creneau in chromosome:
        nb_etudiants = cours_info[cours][0]
        if capacites_salles[salle] < nb_etudiants:
            score -= 30

    # C. Préférences Profs (AJOUTÉ)
    for cours, salle, creneau in chromosome:
        prof_id = cours_info[cours][1]
        jour_actuel = creneau // 3
        
        if jour_actuel == prefs_profs[prof_id]:
            score += 10

    return max(0, score)

# --- 4. CROISEMENT ET MUTATION ---
def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)
    return parent1[:point] + parent2[point:]

def mutation(chromosome, taux=0.1):
    nouveau = chromosome[:] 
    for i in range(len(nouveau)):
        if random.random() < taux:
            cours, salle, creneau = nouveau[i]
            if random.random() < 0.5:
                salle = random.randint(0, NB_SALLES - 1)
            else:
                creneau = random.randint(0, NB_CRENEAUX - 1)
            nouveau[i] = (cours, salle, creneau)
    return nouveau

# --- 5. ALGORITHME GÉNÉTIQUE PRINCIPAL ---
def algo_genetique():
    population = [generer_emploi_aleatoire() for _ in range(100)]
    meilleur_score_global = 0
    meilleure_solution = []
    
    print("Démarrage de l'évolution (avec préférences profs)...")
    
    for generation in range(100): 
        scores = [fitness_emploi_du_temps(ind) for ind in population]
        
        max_score_gen = max(scores)
        index_best = scores.index(max_score_gen)
        
        if max_score_gen > meilleur_score_global:
            meilleur_score_global = max_score_gen
            meilleure_solution = population[index_best]
            print(f"Gen {generation}: Nouveau record = {meilleur_score_global}")
        
        pop_triee = [x for _, x in sorted(zip(scores, population), key=lambda pair: pair[0], reverse=True)]
        parents = pop_triee[:50] 
        
        enfants = []
        while len(enfants) < 50:
            p1, p2 = random.sample(parents, 2)
            enfant = crossover(p1, p2)
            enfant = mutation(enfant)
            enfants.append(enfant)
            
        population = parents + enfants

    return meilleure_solution, meilleur_score_global

# --- 6. EXÉCUTION ET AFFICHAGE AMÉLIORÉ ---

solution_finale, score_final = algo_genetique()

print(f"\n--- RÉSULTAT FINAL (Score: {score_final}) ---")
print("Génération du fichier 'resultat_emploi_du_temps.txt'...")

solution_triee = sorted(solution_finale, key=lambda x: x[2]) 

noms_jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]

with open("resultat_emploi_du_temps.txt", "w", encoding="utf-8") as f:
    
    f.write(f"RAPPORT EMPLOI DU TEMPS - SCORE : {score_final}\n")
    f.write("="*50 + "\n\n")

    jour_actuel = -1
    
    for cours_id, salle_id, creneau in solution_triee:
        nb_eleves = cours_info[cours_id][0]
        capacite = capacites_salles[salle_id]
        prof_id = cours_info[cours_id][1]
        jour_index = creneau // 3
        creneau_jour = creneau % 3 + 1
        
        if jour_index != jour_actuel:
            header = f"\n--- {noms_jours[jour_index]} ---\n"
            print(header)
            f.write(header)
            jour_actuel = jour_index
        
        statut = "✅"
        msg_bonus = ""
        if capacite < nb_eleves: 
            statut = "⚠️"
            msg_bonus += " [TROP PETIT]"
        if jour_index == prefs_profs[prof_id]: 
            statut += " ❤️"
            msg_bonus += " [PREF PROF]"
        
        ligne = (f"  Créneau {creneau_jour} : Cours {cours_id:02d} | "
                 f"Prof {prof_id:02d} | Salle {salle_id:02d} ({nb_eleves}/{capacite}) "
                 f"| {statut}{msg_bonus}")
        
        print(ligne)
        f.write(ligne + "\n")

print("\nTerminé ! Ouvre le fichier 'resultat_emploi_du_temps.txt' pour tout voir.")

Démarrage de l'évolution (avec préférences profs)...
Gen 0: Nouveau record = 520
Gen 1: Nouveau record = 560
Gen 2: Nouveau record = 650
Gen 5: Nouveau record = 680
Gen 6: Nouveau record = 740
Gen 8: Nouveau record = 800
Gen 10: Nouveau record = 810
Gen 14: Nouveau record = 890
Gen 22: Nouveau record = 900
Gen 24: Nouveau record = 910
Gen 26: Nouveau record = 920
Gen 28: Nouveau record = 950
Gen 29: Nouveau record = 960
Gen 30: Nouveau record = 970
Gen 31: Nouveau record = 990
Gen 36: Nouveau record = 1000
Gen 39: Nouveau record = 1030
Gen 41: Nouveau record = 1040
Gen 42: Nouveau record = 1050
Gen 50: Nouveau record = 1060
Gen 51: Nouveau record = 1110
Gen 58: Nouveau record = 1130
Gen 71: Nouveau record = 1150
Gen 78: Nouveau record = 1160
Gen 88: Nouveau record = 1170
Gen 96: Nouveau record = 1180

--- RÉSULTAT FINAL (Score: 1180) ---
Génération du fichier 'resultat_emploi_du_temps.txt'...

--- Lundi ---

  Créneau 1 : Cours 10 | Prof 01 | Salle 01 (40/50) | ✅ ❤️ [PREF PROF]
  Créne